In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
MODEL_PATH = "../models/finrisk_xgboost_model.pkl"
PREPROCESSOR_PATH = "../models/finrisk_preprocessor.pkl"
FEATURES_PATH = "../models/finrisk_feature_names.pkl"

model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)
feature_names = joblib.load(FEATURES_PATH)

print("Model:", type(model))
print("Preprocessor:", type(preprocessor))
print("Number of features:", len(feature_names))

Model: <class 'xgboost.sklearn.XGBClassifier'>
Preprocessor: <class 'sklearn.pipeline.Pipeline'>
Number of features: 70


In [3]:
print("Features used by the model:")

for i, feature in enumerate(feature_names, 1):
    print(f"{i:2}. {feature}")

Features used by the model:
 1. loan_amnt
 2. funded_amnt
 3. funded_amnt_inv
 4. int_rate
 5. installment
 6. policy_code
 7. delinq_amnt
 8. annual_inc
 9. dti
10. mths_since_rcnt_il
11. mths_since_recent_bc
12. mths_since_recent_inq
13. delinq_2yrs
14. inq_last_6mths
15. open_acc
16. pub_rec
17. revol_bal
18. revol_util
19. total_acc
20. collections_12_mths_ex_med
21. acc_now_delinq
22. tot_coll_amt
23. tot_cur_bal
24. open_acc_6m
25. open_act_il
26. open_il_12m
27. open_il_24m
28. total_bal_il
29. il_util
30. open_rv_12m
31. open_rv_24m
32. max_bal_bc
33. all_util
34. total_rev_hi_lim
35. inq_fi
36. total_cu_tl
37. inq_last_12m
38. acc_open_past_24mths
39. avg_cur_bal
40. bc_open_to_buy
41. bc_util
42. chargeoff_within_12_mths
43. mo_sin_old_il_acct
44. mo_sin_old_rev_tl_op
45. mo_sin_rcnt_rev_tl_op
46. mo_sin_rcnt_tl
47. mort_acc
48. num_accts_ever_120_pd
49. num_actv_bc_tl
50. num_actv_rev_tl
51. num_bc_sats
52. num_bc_tl
53. num_il_tl
54. num_op_rev_tl
55. num_rev_accts
56. num_

In [4]:
import pandas as pd

DATA_PATH = "../data/raw/finrisk_ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1303638, 73)

Columns:
['loan_id', 'borrower_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'policy_code', 'delinq_amnt', 'target', 'annual_inc', 'dti', 'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_inq', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal

In [5]:
print("Target columns:")
print([col for col in df.columns if "target" in col.lower() or "default" in col.lower()])

Target columns:
['target']


In [6]:
print(df["target"].value_counts(dropna=False))

target
0    1041952
1     261686
Name: count, dtype: int64


In [7]:
missing_features = [
    feature for feature in feature_names
    if feature not in df.columns
]

print("Missing model features:", missing_features)
print("Number missing:", len(missing_features))

Missing model features: []
Number missing: 0


In [8]:
X = df[feature_names].copy()
y = df["target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1303638, 70)
y shape: (1303638,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training shape: (1042910, 70)
Test shape: (260728, 70)

Training target distribution:
target
0    0.799265
1    0.200735
Name: proportion, dtype: float64

Test target distribution:
target
0    0.799266
1    0.200734
Name: proportion, dtype: float64


In [10]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("XGBoost Model Performance")
print("--------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)

XGBoost Model Performance
--------------------------
Accuracy : 0.6609992022337455
Precision: 0.33155779833660404
Recall   : 0.6779142862601983
F1 Score : 0.4453174518503643
ROC-AUC  : 0.7292238074983448


In [12]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[136861  71530]
 [ 16857  35480]]


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.66      0.76    208391
           1       0.33      0.68      0.45     52337

    accuracy                           0.66    260728
   macro avg       0.61      0.67      0.60    260728
weighted avg       0.78      0.66      0.69    260728

